# 부록B. 점포 월 매출 예측: 머신러닝

- **분석 배경**
  - 선형 회귀 실습에서는 월 매출이 어떤 특성에 따라 달라지는지 해석했습니다.
  - 이번에는 전월 말 시점에 알 수 있는 정보로 다음 달 점포 월 매출을 미리 예측해, 매출 계획의 기준선으로 쓰고자 합니다.
  - 머신러닝 알고리즘인 회귀 나무와 랜덤 포레스트로 예측 모델을 만들고, 선형 회귀(벤치마크 모델)와 성능을 비교합니다.
- **분석 대상**
  - 100개 점포의 월 매출입니다.
  - 2024 ~ 2025년으로 학습하고, 2026년 1 ~ 6월로 모델을 고른 뒤, 2026년 7 ~ 10월로 한 번 평가합니다.
- **분석 질문**
  - 다음 달 점포 월 매출을 얼마나 정확하게 예측할 수 있는가?
- **사용 데이터**
  - `store_month_clean.pkl`(훈련셋: 2024 ~ 2025년 점포 × 월 2,400행)
  - `store_month_valid.csv`(검증셋: 2026년 1 ~ 6월 점포 × 월 600행)
  - `store_month_test.csv`(시험셋: 2026년 7 ~ 10월 점포 × 월 400행)
- **분석 단위**: 점포 × 월

**학습 목표**

1. 예측 시점에 알 수 없는 정보(데이터 누수)를 구분하고, 시간 기준으로 데이터를 나눌 수 있다.
2. 결정계수·MSE·RMSE·MAE·MAPE로 회귀 모델을 평가하고, 기준 모델·벤치마크 모델과 비교할 수 있다.
3. 회귀 나무의 과적합을 진단하고, 비용 복잡도 가지치기로 줄일 수 있다.
4. 검증셋 기준 그리드 서치로 랜덤 포레스트의 하이퍼파라미터를 고를 수 있다.
5. 검증셋으로 최종 모델을 정하고, 시험셋으로 한 번 평가할 수 있다.
6. 특성 중요도, 순열 중요도와 부분 의존도(PDP)로 모델이 월 매출을 어떻게 예측하는지 설명할 수 있다.

## 0. 준비

### 라이브러리 불러오기

In [ ]:
# 필요한 모듈을 임포트합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import hds

### 그래프 한글 폰트 설정

구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.<br>
음수 부호(−)가 네모로 깨지지 않도록 하는 설정도 함께 처리합니다.

In [ ]:
# 구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.
# 처음 실행할 때만 인터넷 연결이 필요하고, 이후에는 내려받은 파일을 다시 사용합니다.
hds.plot.set_font(font_name='Gowun Batang')

# 그래프 제목을 굵게 표시합니다.
plt.rc(group='axes', titleweight='bold')

### 난수 시드

무작위 과정이 들어가는 코드가 매번 같은 결과를 내도록 시드를 고정합니다.

In [ ]:
# 난수 시드를 SEED에 할당합니다.
SEED = 42

### 데이터 불러오기

기술통계 실습에서 점포 마스터를 결합하고 매출 입력 오류를 보정해 저장한 pkl 파일을 불러옵니다.<br>
pkl 파일이 없으면 기술통계 실습 노트북을 먼저 실행합니다.

In [ ]:
# 기술통계 실습에서 저장한 pkl 파일을 읽고 train_df에 할당합니다.
train_df = pd.read_pickle(filepath_or_buffer='../data/store_month_clean.pkl')

# train_df의 처음 5행과 마지막 5행을 확인합니다.
# [참고] 데이터프레임 이름만 실행하면 처음 5행과 마지막 5행, 행 개수와 열 개수를 함께 보여줍니다.
train_df

In [ ]:
# train_df의 정보를 확인합니다.
train_df.info()

입력 오류는 기술통계 실습에서 보정했습니다.<br>
반면 선형 회귀 실습에서 찾은 영향점(영업일수가 크게 줄어든 달)은 **실제로 일어난 사건**이므로 유지합니다.<br>
예측 모델은 앞으로도 이런 달이 생길 수 있다는 것을 알고 있어야 하기 때문입니다.

모델을 고르고 성능을 평가할 2026년 데이터를 검증셋(1 ~ 6월)과 시험셋(7 ~ 10월) 두 파일로 받았습니다.<br>
두 파일은 훈련 데이터와 열 구성이 같으므로 세 데이터를 위아래로 붙여 데이터를 비교할 때 사용합니다.

In [ ]:
# 검증셋 점포 월 매출 csv 파일을 읽고 valid_df에 할당합니다.
valid_df = pd.read_csv(filepath_or_buffer='../data/store_month_valid.csv', encoding='utf-8-sig')

# valid_df의 처음 5행과 마지막 5행을 확인합니다.
valid_df

In [ ]:
# valid_df의 정보를 확인합니다.
valid_df.info()

In [ ]:
# 시험셋 점포 월 매출 csv 파일을 읽고 test_df에 할당합니다.
test_df = pd.read_csv(filepath_or_buffer='../data/store_month_test.csv', encoding='utf-8-sig')

# test_df의 처음 5행과 마지막 5행을 확인합니다.
test_df

In [ ]:
# test_df의 정보를 확인합니다.
test_df.info()

## 1. 문제 정의

| 구분 | 선형 회귀 | 예측 모델 |
|---|---|---|
| 질문 | 어떤 특성이 점포 월 매출과 얼마나 관련 있는가? [**설명**] | 아직 오지 않은 달의 매출은 얼마인가? [**예측**] |
| 결과물 | 회귀계수, 신뢰구간, 유의확률 | 점포별 다음 달 매출 예측값 |
| 평가 | 계수의 방향·크기·유의성, 잔차 진단 | 학습에 쓰지 않은 기간의 매출 예측 정확도(RMSE, MAE, MAPE 등) |
| 활용 | 점포 매출과 관련된 특성 파악 | 점포 운영 계획 수립 |

예측 모델에서는 '왜'보다 '얼마나 정확한가'가 먼저이므로, **학습에 쓰지 않은 데이터**로 평가하는 것이 핵심입니다.

**데이터 분할**

| 구분 | 기간 | 용도 |
|---|---|---|
| 훈련 | 2024-01 ~ 2025-12 | 모델 학습 |
| 검증 | 2026-01 ~ 2026-06 | 가지치기 강도·하이퍼파라미터와 최종 모델 선택 (**교차검증 대신 미리 정한 검증셋 사용**) |
| 시험 | 2026-07 ~ 2026-10 | 마지막에 한 번 성능 확인 |

- 실제로 모델은 **과거의 매출로 학습해서 다음 달 매출을 예측**합니다.
  - 여러 모델 중 최종 모델을 선택하고, 최종 모델을 평가하는 것도 훈련 기간 이후의 달을 대상으로 합니다.
- 데이터를 무작위로 나누지 않고 시간 순서대로 나누는 이유는 점포 매출이 시간이 지나며 생기는 변화를 반영해야 하기 때문입니다.
  - 훈련셋과 검증셋을 무작위로 나누면 미래 데이터로 모델을 학습하고, 과거 데이터로 최종 모델을 선택하는 경우가 발생합니다.

**질문**: 입력변수와 매출의 분포는 훈련셋, 검증셋, 시험셋에서 비슷한가?

In [ ]:
# 세 데이터를 위아래로 붙이고, 데이터 구분을 dataset 열로 추가해서 all_sets에 할당합니다.
# [참고] keys=['훈련', '검증', '시험']는 각 데이터를 구분하는 인덱스이고, names=['dataset']은 그 인덱스의 이름입니다.
# [참고] reset_index()는 데이터프레임의 인덱스를 첫 번째 열로 추가하며, 인덱스를 초기화합니다.
all_sets = pd.concat(objs=[train_df, valid_df, test_df], keys=['훈련', '검증', '시험'], names=['dataset']) \
    .reset_index(level='dataset')

# 데이터별 행 개수를 확인합니다.
all_sets['dataset'].value_counts(sort=False)

**해석**
- 2024 ~ 2025년 2년치(2,400행)로 학습하고, 2026년 1 ~ 6월(600행)로 최종 모델을 선택하고, 2026년 7 ~ 10월(400행)로 성능을 평가합니다.
  - 학습 기간이 2년이면 설·추석이나 여름 같은 계절 패턴을 두 번씩 볼 수 있습니다.

In [ ]:
# 0과 1로 된 변수를 포함한 수치형 입력변수와 목표변수 이름을 리스트로 생성합니다.
# [참고] 점포 특성(면적, 품목 수 등)은 같은 점포라 기간이 달라도 값이 같으므로 제외합니다.
num_cols = ['open_days', 'weekend_days', 'holiday_days', 'holiday_season', 'event_days', 'sales']

# 데이터별 수치형 입력변수와 목표변수의 평균을 계산하고 행과 열을 바꿔서 확인합니다.
# [참고] sort=False는 집단을 이름순으로 정렬하지 않고 데이터에 나온 순서대로(훈련, 검증, 시험 순으로) 보여줍니다.
all_sets.groupby(by='dataset', sort=False)[num_cols].mean().T.rename_axis(columns=None).round(decimals=2)

In [ ]:
# 검증셋 기간과 같은 1 ~ 6월 데이터만 남기고 valid_months에 할당합니다.
valid_months = all_sets.loc[all_sets['ym'].str[5:7].between(left='01', right='06'), :]

# 연도별 1 ~ 6월 입력변수와 매출의 평균을 계산하고 행과 열을 바꿔서 확인합니다.
valid_months.groupby(by=valid_months['ym'].str[:4])[num_cols].mean().T.round(decimals=2)

In [ ]:
# 시험셋 기간과 같은 7 ~ 10월 데이터만 남기고 test_months에 할당합니다.
test_months = all_sets.loc[all_sets['ym'].str[5:7].between(left='07', right='10'), :]

# 연도별 7 ~ 10월 입력변수와 매출의 평균을 계산하고 행과 열을 바꿔서 확인합니다.
test_months.groupby(by=test_months['ym'].str[:4])[num_cols].mean().T.round(decimals=2)

**해석**
- 데이터별 평균을 그대로 비교하면 `holiday_season`(명절 여부)와 `open_days`(영업일수)에서 조금씩 차이를 보입니다.
  - 훈련셋을 검증셋·시험셋과 같은 달끼리 비교하면 명절 여부는 1 ~ 6월이 모두 0.17이고, 7 ~ 10월이 모두 0.25이므로 연도마다 같습니다.
  - 시험셋(7 ~ 10월)은 추석이 든 달이 넉 달 중 하나라서 열두 달 중 두 번인 훈련셋보다 비율이 높았던 것입니다.
- 시간 순서로 나눈 데이터에서 기간 길이나 계절이 다르면, 단순 평균의 차이는 분포 변화가 아니라 **어느 달이 들어 있는지의 차이**일 수 있습니다.
  - 같은 달끼리 비교해야 실제로 분포가 바뀌었는지 판단할 수 있습니다. [데이터 드리프트]
- 같은 달끼리 보면 입력변수와 매출 평균이 연도마다 비슷하므로 2026년 데이터도 학습 기간과 비슷한 조건이라고 볼 수 있습니다.

## 2. 사용할 수 없는 정보 확인 (데이터 누수)

예측 시점(전월 말)에 **알 수 있는 정보**만 입력변수로 쓸 수 있습니다.

| 입력변수 | 예측 시점에 알 수 있나? | 처리 |
|---|---|---|
| `transactions`, `avg_ticket` | 아니오. 같은 달 매출과 함께 생기는 **결과**임 | 제외 |
| `rain_days`, `extreme_days` | 아니오. 그 달이 지나야 알 수 있는 실제 날씨 데이터임 | 기본 모형에서 제외 |
| `event_days`, `open_days`, `weekend_days`,<br>`holiday_days`, `holiday_season` | 예. 달력과 행사 계획으로 미리 정해짐 | 사용 |
| 점포 특성(면적, 품목 수, 주차면수, 유형, 지역, 휴업 요일) | 예. 미리 알고 있음 | 사용 |

[참고] 날씨를 쓰고 싶다면 실제 강수일수가 아니라 **예측 시점의 기상 예보값**을 써야 합니다. 현재 데이터에는 예보값이 없으므로 날씨를 제외합니다.

**질문**: `transactions`(구매 건수)는 매출과 얼마나 관련 있는가?

In [ ]:
# 구매 건수와 매출의 Pearson 상관계수를 확인합니다.
print(f'구매 건수와 매출의 상관계수: {all_sets["transactions"].corr(other=all_sets["sales"]):.3f}')

**해석**
- 구매 건수는 매출과 거의 같이 움직입니다.
- 매출 = 구매 건수 × 객단가이므로 당연합니다.
- 이 변수를 넣으면 성능이 아주 좋아 보이지만, 다음 달의 구매 건수는 매출만큼이나 모르는 값입니다.
- 8절에서 실제로 확인합니다.

## 3. 입력변수

**질문**: 예측 시점에 알 수 있는 변수로 입력변수 행렬(X)과 목표변수 벡터(y)를 만듭니다.<br>
범주형은 원-핫 인코딩합니다.

In [ ]:
# 훈련셋의 연월에서 월(1 ~ 12)을 추출해 month 열로 추가합니다.
# [참고] 계절 패턴을 반영하기 위함입니다.
train_df['month'] = train_df['ym'].str[5:7].astype(dtype=int)

# 수치형 입력변수 이름을 리스트로 생성합니다.
num_features = ['open_days', 'weekend_days', 'holiday_days', 'holiday_season', 'event_days', 'month', 'floor_area', 
                'sku_count', 'parking']

# 범주형 입력변수 이름을 리스트로 생성합니다.
cat_features = ['store_type', 'region', 'closure_day']

# 훈련셋의 범주형 변수를 원-핫 인코딩하고 X_train에 할당합니다.
X_train = pd.get_dummies(data=train_df[num_features + cat_features], columns=cat_features, dtype=float)

# 훈련셋의 모든 열을 실수형으로 변경합니다.
# [참고] 부분 의존도(PDP)는 정수형 열을 지원하지 않으므로 모든 열을 실수형으로 바꿉니다.
X_train = X_train.astype(dtype=float)

# 훈련셋의 목표변수 벡터를 생성합니다.
y_train = train_df['sales'].copy()

In [ ]:
# 검증셋의 연월에서 월(1 ~ 12)을 추출해 month 열로 추가합니다.
valid_df['month'] = valid_df['ym'].str[5:7].astype(dtype=int)

# 검증셋의 입력변수 행렬을 같은 방법으로 생성하고 X_valid에 할당합니다.
X_valid = pd.get_dummies(data=valid_df[num_features + cat_features], columns=cat_features, dtype=float)

# X_valid의 모든 열을 실수형으로 변경하고, 열 순서를 훈련셋과 맞춥니다.
# [참고] fill_value=0은 어떤 범주가 없어서 더미 변수가 빠진 것을 0으로 채웁니다.
X_valid = X_valid.astype(dtype=float).reindex(columns=X_train.columns, fill_value=0)

# 검증셋의 목표변수 벡터를 생성합니다.
y_valid = valid_df['sales'].copy()

In [ ]:
# 시험셋의 연월에서 월(1 ~ 12)을 추출해 month 열로 추가합니다.
test_df['month'] = test_df['ym'].str[5:7].astype(dtype=int)

# 시험셋의 입력변수 행렬을 같은 방법으로 생성하고 X_test에 할당합니다.
X_test = pd.get_dummies(data=test_df[num_features + cat_features], columns=cat_features, dtype=float)

# X_test의 모든 열을 실수형으로 변경하고, 열 순서를 훈련셋과 맞춥니다.
X_test = X_test.astype(dtype=float).reindex(columns=X_train.columns, fill_value=0)

# 시험셋의 목표변수 벡터를 생성합니다.
y_test = test_df['sales'].copy()

In [ ]:
# 훈련셋, 검증셋, 시험셋의 크기를 확인합니다.
print('훈련셋 크기:', X_train.shape)
print('검증셋 크기:', X_valid.shape)
print('시험셋 크기:', X_test.shape)

In [ ]:
# 입력변수 이름을 확인합니다.
X_train.columns.tolist()

**해석**
- 선형 회귀 실습에서는 계수를 해석하기 위해 매출과 면적에 로그를 취하고, 면적과 정보가 겹치는 품목 수를 뺐습니다.
  - 이번 실습에서는 계수를 해석하지 않고 예측만 하므로 원래 값을 그대로 쓰고 면적과 품목 수도 함께 넣습니다.
- 의사결정나무는 변수 값의 크기를 비교하여 데이터를 나누므로 연속형 변수는 그대로, 범주형 변수는 0/1 열로 바꿔서 넣습니다.
  - 나무 모델은 값의 크기 순서로만 나누므로 로그를 취해도 분할 결과가 같고, 다중공선성도 예측에는 문제가 되지 않습니다.

## 4. 평가 함수, 기준 모델, 벤치마크 모델

**질문**: 여러 모델을 같은 지표로 비교하기 위해 성능 지표를 한 번에 계산하는 함수와, 여러 모델의 결과를 한 표로 모으는 함수를 만듭니다.

In [ ]:
# 실제값과 예측값을 지정하면 5가지 성능 지표를 딕셔너리로 반환하는 함수를 생성합니다.
# [참고] r2_score(), mean_squared_error(), root_mean_squared_error(), mean_absolute_error(), mean_absolute_percentage_error()는 
# sklearn.metrics 모듈에 포함되어 있습니다.
# [참고] mean_absolute_percentage_error()는 비율을 반환하므로 100을 곱해 %로 바꿉니다.
def score(y, pred):
    return {'결정계수': r2_score(y_true=y, y_pred=pred),
            'MSE': mean_squared_error(y_true=y, y_pred=pred),
            'RMSE': root_mean_squared_error(y_true=y, y_pred=pred),
            'MAE': mean_absolute_error(y_true=y, y_pred=pred),
            'MAPE(%)': mean_absolute_percentage_error(y_true=y, y_pred=pred) * 100}

# 모델과 입력변수 행렬, 목표변수를 지정하면 예측값을 계산해 5가지 성능 지표를 딕셔너리로 반환하는 함수를 생성합니다.
def evaluate(model, X, y):
    return score(y=y, pred=model.predict(X=X))

# 모델 이름과 모델을 담은 딕셔너리를 지정하면 모델별 성능 지표를 데이터프레임으로 반환하는 함수를 생성합니다.
# [참고] 행은 모델 이름, 열은 성능 지표입니다.
def compare(models, X, y):
    scores = {name: evaluate(model, X, y) for name, model in models.items()}
    return pd.DataFrame(data=scores).T

**해석**
- 회귀 모델의 성능 지표는 다음과 같습니다.

  | 지표 | 뜻 | 특징 |
  |---|---|---|
  | **결정계수(R²)** | 매출 변동 중 모델이 설명하는 비율 | 1에 가까울수록 좋음<br>회귀 모델의 성능 지표 중 유일하게 점수가 높을수록 좋은 지표 |
  | **MSE** | 오차 제곱의 평균 | 오차를 제곱하여 평균을 낸 것으로, 오차가 클수록 벌점을 크게 부여함<br>단위가 '백만 원의 제곱'이 되어 해석하기 어려움 |
  | **RMSE** | MSE의 제곱근 | 오차의 크기를 원래 단위(백만 원)로 되돌린 것으로, 해석하기 쉬움 |
  | **MAE** | 오차 절댓값의 평균 | 오차의 크기를 평균낸 것, 평균적으로 몇 백만 원 틀리는가 |
  | **MAPE** | 실제값 대비 오차 비율의 평균 | 오차의 비율을 평균낸 것, 점포 규모가 달라도 비교할 수 있어 현업 보고에 많이 씀 |

**질문**: 가장 단순한 예측(모든 달의 매출을 훈련셋 평균으로 예측)은 얼마나 정확한가?

In [ ]:
# 훈련셋의 평균 매출로 예측하는 기준 모델을 생성합니다.
# [참고] DummyRegressor 클래스는 sklearn.dummy 모듈에 포함되어 있습니다.
dummy = DummyRegressor(strategy='mean')

# 훈련셋으로 기준 모델을 학습합니다.
dummy.fit(X=X_train, y=y_train)

In [ ]:
# 성능을 비교할 모델을 저장할 빈 딕셔너리를 생성합니다.
models = {}

# 기준 모델을 models에 추가합니다.
models['기준 모델'] = dummy

# 지금까지 추가한 모델의 검증 성능을 소수점 3자리까지 반올림해서 확인합니다.
compare(models, X_valid, y_valid).round(decimals=3)

**해석**
- 기준 모델은 점포와 달에 상관없이 모든 매출을 훈련셋 평균(약 3,599백만 원)으로 예측합니다.
- 결정계수가 0이므로 매출 변동을 전혀 설명하지 못합니다.
  - 결정계수의 -0.000은 검증셋 평균이 훈련셋 평균과 조금 달라서 생긴 것으로, 0과 같다고 봅니다.
- MAPE는 약 35%로, 평균적으로 실제 매출의 3분의 1 정도를 틀립니다.
- 모든 모델은 최소한 이 기준 모델보다 나아야 합니다.

**질문**: 점포마다 2025년 월평균 매출로 예측하면(점포별 평균) 얼마나 정확한가?<br>
예측 대상은 과거 매출을 이미 알고 있는 같은 100개 점포이므로, 각 점포의 평소 매출 수준을 그대로 쓰는 것이 더 현실적인 기준선입니다.

In [ ]:
# 훈련셋의 2025년 행만 골라 점포별 월평균 매출을 계산하고 store_mean에 할당합니다.
# [참고] 훈련 기간의 매출만 사용하므로 검증·시험 기간의 정보가 섞이지 않습니다.
store_mean = train_df.loc[train_df['ym'].str.startswith('2025'), :].groupby(by='store_id')['sales'].mean()

In [ ]:
# 검증셋과 시험셋의 점포 ID에 점포별 평균을 붙여 예측값을 만들고 valid_store_pred, test_store_pred에 할당합니다.
valid_store_pred = valid_df['store_id'].map(func=store_mean)
test_store_pred = test_df['store_id'].map(func=store_mean)

In [ ]:
# 기준 모델과 점포별 평균의 검증 성능을 소수점 3자리까지 반올림해서 비교합니다.
# [참고] 점포별 평균은 모델 객체가 아니라 예측값이므로 score()로 직접 계산합니다.
pd.DataFrame(data={'기준 모델': evaluate(dummy, X_valid, y_valid), 
                   '점포별 평균': score(y=y_valid, pred=valid_store_pred)}) \
    .T.round(decimals=3)

**해석**
- 점포별 평균은 각 점포의 2025년 월평균 매출을 그 점포의 2026년 매출 예측값으로 씁니다.
- 결정계수 0.958, MAPE 약 6.2%로 기준 모델(MAPE 약 35%)보다 훨씬 정확합니다.
  - 점포마다 매출 수준이 크게 다르므로, 각 점포의 평소 수준만 알아도 매출의 대부분을 맞힐 수 있습니다.
  - 기준 모델 대비 개선의 상당 부분은 '점포마다 매출 수준이 다르다'는 것을 아는 데서 나옵니다.
- 머신러닝 모델이 실무에서 쓸 가치가 있는지는 이 점포별 평균보다 얼마나 나은지로 판단합니다.
  - 점포별 평균은 모델 객체가 아니라 예측값이므로 `models`에 넣지 않고, 최종 모델을 고른 뒤 따로 비교합니다.

**질문**: 입력변수를 그대로 넣은 선형 회귀로 매출을 예측하면 성능이 어떤가?<br>
머신러닝 모델은 이 성능보다 나아야 쓸 이유가 있습니다.

In [ ]:
# 선형 회귀 모델을 생성합니다.
# [참고] LinearRegression 클래스는 sklearn.linear_model 모듈에 포함되어 있습니다.
# [참고] 더미 변수를 모두 넣고 면적과 품목 수도 함께 넣어 다중공선성이 있지만, 예측값은 영향을 받지 않으므로 예측용 벤치마크로는 그대로 사용합니다.
# 다만 계수 해석은 기준 범주를 빼고 변수를 고른 선형 회귀 실습의 결과로 합니다.
linear = LinearRegression()

# 훈련셋으로 선형 회귀 모델을 학습합니다.
linear.fit(X=X_train, y=y_train)

In [ ]:
# 선형 회귀를 models에 추가합니다.
models['선형 회귀'] = linear

# 지금까지 추가한 모델의 검증 성능을 소수점 3자리까지 반올림해서 확인합니다.
compare(models, X_valid, y_valid).round(decimals=3)

**해석**
- 선형 회귀는 기준 모델보다 모든 지표가 크게 좋습니다.
  - 이 성능이 머신러닝 모델이 넘어야 할 **벤치마크**입니다.
  - 다만 검증 MAPE는 7.277%로 점포별 평균(6.224%)보다 높아, 입력변수를 그대로 넣은 선형 회귀는 점포의 평소 매출 수준만큼도 맞히지 못합니다.
- 선형 회귀는 각 변수의 효과가 일정하게 더해진다고 가정합니다. [선형 결합]
  - 점포 규모에 따라 다른 변수의 효과가 커지는 상호작용 관계는 입력변수에 직접 넣어 주지 않으면 모델이 스스로 잡지 못합니다.
- 선형 회귀는 계수로 효과를 설명할 수 있으므로 성능이 비슷하다면 선형 회귀가 더 좋은 선택일 수 있습니다.
  - [참고] 선형 회귀 실습처럼 매출과 면적에 로그를 취하면 선형 회귀의 성능도 크게 좋아져, 이번 실습의 랜덤 포레스트와 비슷한 수준이 됩니다.
  - 데이터 구조에 맞게 변수를 처리하면 단순한 모형도 강해질 수 있습니다.

## 5. 회귀 나무: 과적합 진단과 사후 가지치기

**질문**: 깊이 제한 없이 끝까지 자란 회귀 나무는 훈련·검증 성능이 어떤가?

In [ ]:
# 깊이 제한이 없는 회귀 나무 모델을 생성합니다.
# [참고] DecisionTreeRegressor 클래스는 sklearn.tree 모듈에 포함되어 있습니다.
full_tree = DecisionTreeRegressor(random_state=SEED)

# 훈련셋으로 회귀 나무를 학습합니다.
full_tree.fit(X=X_train, y=y_train)

In [ ]:
# 회귀 나무의 리프 노드 개수와 깊이를 확인합니다.
print('리프 노드 개수:', full_tree.get_n_leaves(), '/ 깊이:', full_tree.get_depth())

# 회귀 나무의 훈련 결정계수를 확인합니다.
# [참고] 회귀 모델의 score()는 결정계수(R²)를 반환합니다.
print('훈련 결정계수:', round(number=full_tree.score(X=X_train, y=y_train), ndigits=4))

In [ ]:
# 가지치기 전 회귀 나무를 models에 추가합니다.
models['회귀 나무(가지치기 전)'] = full_tree

# 지금까지 추가한 모델의 검증 성능을 소수점 3자리까지 반올림해서 확인합니다.
compare(models, X_valid, y_valid).round(decimals=3)

**해석**
- 훈련 결정계수가 거의 1입니다.
  - 리프 노드 개수가 훈련셋 행 개수(2,400)에 가까우므로 훈련셋을 외워 버린 셈입니다.
- 학습에 쓰이지 않은 검증셋에 대해서는 선형 회귀보다도 모든 성능 지표가 큽니다.
  - 이것이 **과적합**입니다.
- 리프 노드마다 관측값이 거의 한 개씩이라 분산이 0이고, 그래서 MSE와 RMSE가 특히 큽니다.

**질문**: 끝까지 키운 나무의 가지를 잘라 내면(사후 가지치기) 검증 성능이 좋아지는가?

비용 복잡도 가지치기는 나무의 오차에 리프 노드 개수만큼 벌점을 더한 값이 가장 작은 나무를 고릅니다.<br>
벌점의 크기는 `ccp_alpha`(비용 복잡도 파라미터)로 정합니다.<br>
`ccp_alpha`가 0이면 끝까지 자란 나무이고, 클수록 리프 노드가 적은 작은 나무가 됩니다.<br>
`ccp_alpha` 후보마다 나무를 학습하고, 검증 MAPE가 가장 낮은 값을 고릅니다.<br>
시간 순서가 있는 데이터라 재구매 예측 실습처럼 무작위로 나누는 교차검증 대신, 미리 확정된 검증셋으로 고릅니다.

In [ ]:
# ccp_alpha(비용 복잡도 파라미터) 후보를 계산합니다.
pruning_path = full_tree.cost_complexity_pruning_path(X=X_train, y=y_train)

# 비용 복잡도 파라미터 후보를 ccp_alphas에 할당합니다.
# [참고] 마지막 값은 뿌리 하나만 남는 나무이므로 제외합니다.
ccp_alphas = pruning_path.ccp_alphas[:-1]

# 비용 복잡도 파라미터 후보의 개수를 확인합니다.
print('비용 복잡도 파라미터 후보:', len(ccp_alphas), '개')

In [ ]:
# 가지치기 결과를 저장할 빈 리스트를 생성합니다.
prune_rows = []

# 비용 복잡도 파라미터별로 회귀 나무를 학습하고 리프 노드 개수와 훈련·검증 MAPE를 prune_rows에 추가합니다.
for alpha in ccp_alphas:
    tree = DecisionTreeRegressor(ccp_alpha=alpha, random_state=SEED)
    tree.fit(X=X_train, y=y_train)
    train_mape = evaluate(tree, X_train, y_train)['MAPE(%)']
    valid_mape = evaluate(tree, X_valid, y_valid)['MAPE(%)']
    prune_rows.append({'ccp_alpha': alpha, '훈련 MAPE(%)': train_mape, '검증 MAPE(%)': valid_mape})

In [ ]:
# 가지치기 결과를 데이터프레임으로 변환하고 prune_table에 할당합니다.
prune_table = pd.DataFrame(data=prune_rows)

# 검증 MAPE가 가장 낮은 행을 best_prune에 할당합니다.
best_prune = prune_table.loc[prune_table['검증 MAPE(%)'].idxmin(), :]

# 비용 복잡도 파라미터에 따른 훈련·검증 MAPE를 선 그래프로 시각화하고, 선택한 파라미터를 세로선으로 표시합니다.
plt.figure(figsize=(6, 4), dpi=100)
plt.plot(prune_table['ccp_alpha'], prune_table['훈련 MAPE(%)'], label='훈련')
plt.plot(prune_table['ccp_alpha'], prune_table['검증 MAPE(%)'], label='검증')
plt.axvline(x=best_prune['ccp_alpha'], color='red', linestyle='--', label='선택한 파라미터')
plt.xscale(value='log')
plt.title(label='비용 복잡도 파라미터에 따른 훈련·검증 MAPE')
plt.xlabel(xlabel='비용 복잡도 파라미터 ccp_alpha (로그 척도, 오른쪽일수록 작은 나무)')
plt.ylabel(ylabel='MAPE (%)')
plt.legend()
plt.show()

In [ ]:
# 선택한 비용 복잡도 파라미터와 훈련·검증 MAPE를 확인합니다.
print('선택한 ccp_alpha:', round(number=best_prune['ccp_alpha'], ndigits=5))
print('훈련 MAPE(%):', round(number=best_prune['훈련 MAPE(%)'], ndigits=3))
print('검증 MAPE(%):', round(number=best_prune['검증 MAPE(%)'], ndigits=3))

In [ ]:
# 선택한 비용 복잡도 파라미터로 회귀 나무 모델을 생성하고 pruned_tree에 할당합니다.
pruned_tree = DecisionTreeRegressor(ccp_alpha=best_prune['ccp_alpha'], random_state=SEED)

# 훈련셋으로 가지치기한 회귀 나무를 학습합니다.
pruned_tree.fit(X=X_train, y=y_train)

# 가지치기한 회귀 나무의 리프 노드 개수와 깊이를 확인합니다.
print('리프 노드 개수:', pruned_tree.get_n_leaves())
print('나무 깊이:', pruned_tree.get_depth())

In [ ]:
# 가지치기한 회귀 나무를 models에 추가합니다.
models['회귀 나무(가지치기 후)'] = pruned_tree

# 지금까지 추가한 모델의 검증 성능을 소수점 3자리까지 반올림해서 확인합니다.
compare(models, X_valid, y_valid).round(decimals=3)

**해석**
- 비용 복잡도 파라미터가 작을수록(왼쪽) 훈련 MAPE는 0에 가깝지만, 검증 MAPE는 훈련 MAPE와 일정한 간격을 유지합니다.
  - 파라미터를 키우면 검증 MAPE가 감소하다가, 너무 크면 나무가 지나치게 작아져 다시 증가합니다.
  - 검증 MAPE가 가장 낮은 지점이 과소적합과 과적합 사이의 균형점입니다.
- 가지치기로 리프 노드 개수가 크게 줄었고, 검증 MAPE도 가지치기 전보다 낮아졌습니다.
  - 가지치기는 나무를 끝까지 키운 뒤 불필요한 노드를 잘라내므로 깊이를 미리 정하는 방식보다 중요한 가지를 남기기 쉽습니다.
- 검증 MAPE는 선형 회귀와 비슷하지만, RMSE와 MAE는 선형 회귀보다 여전히 큽니다.
  - 나무 하나는 같은 리프 노드에 들어간 행을 모두 같은 값으로 예측하므로 매출처럼 연속적으로 변하는 값을 계단 모양으로만 맞힐 수 있기 때문입니다.

**질문**: 가지치기한 회귀 나무가 매출을 예측하려고 데이터를 나누는 규칙을 읽어 봅니다.

In [ ]:
# 가지치기한 회귀 나무의 분할 규칙을 텍스트로 출력합니다.
# [참고] export_text()는 sklearn.tree 모듈에 포함되어 있습니다.
# [참고] max_depth=3은 모델은 그대로 두고, 출력만 위에서부터 3단계까지 보여줍니다.
print(export_text(decision_tree=pruned_tree, feature_names=X_train.columns.tolist(), max_depth=3, decimals=2))

## 6. 랜덤 포레스트와 그리드 서치

**질문**: 랜덤 포레스트의 하이퍼파라미터 조합을 모두 시도해 보고(그리드 서치), 검증 MAPE가 가장 낮은 조합을 고릅니다.

- `max_features`: 각 분할에서 고려할 변수 비율. 작을수록 나무들이 서로 달라집니다.
- `min_samples_leaf`: 리프 노드 하나에 들어가야 하는 최소 행 개수. 클수록 나무가 단순해집니다.

시간 순서가 있는 데이터라 무작위로 나누는 교차검증 대신, 검증 기간 하나로 조합을 비교합니다.

In [ ]:
# 랜덤 포레스트 하이퍼파라미터 후보를 딕셔너리로 생성합니다.
param_grid = {'max_features': [0.3, 0.5, 0.7], 'min_samples_leaf': [3, 5, 7, 10]}

# 하이퍼파라미터 후보의 모든 조합을 리스트로 생성합니다.
# [참고] ParameterGrid 클래스는 sklearn.model_selection 모듈에 포함되어 있습니다.
param_list = list(ParameterGrid(param_grid=param_grid))

# 하이퍼파라미터 조합의 개수를 확인합니다.
print('조합 개수:', len(param_list))

In [ ]:
# 그리드 서치 결과를 저장할 빈 리스트를 생성합니다.
grid_search = []

# 조합마다 랜덤 포레스트를 학습하고 훈련·검증 MAPE를 grid_search에 추가합니다.
# [참고] RandomForestRegressor 클래스는 sklearn.ensemble 모듈에 포함되어 있습니다.
for params in param_list:
    rf = RandomForestRegressor(n_estimators=300, random_state=SEED, **params)
    rf.fit(X=X_train, y=y_train)
    train_mape = evaluate(rf, X_train, y_train)['MAPE(%)']
    valid_mape = evaluate(rf, X_valid, y_valid)['MAPE(%)']
    grid_search.append({**params, '훈련 MAPE(%)': train_mape, '검증 MAPE(%)': valid_mape})

In [ ]:
# 그리드 서치 결과를 검증 MAPE 오름차순으로 정렬하고 grid_table에 할당합니다.
grid_table = pd.DataFrame(data=grid_search).sort_values(by='검증 MAPE(%)')

# 그리드 서치 결과를 소수점 3자리까지 반올림해서 확인합니다.
grid_table.round(decimals=3)

In [ ]:
# 검증 MAPE가 가장 낮은 조합을 best_params에 할당합니다.
# [참고] grid_table의 인덱스는 param_list의 순서와 같습니다.
best_params = param_list[grid_table.index[0]]

# 선택한 하이퍼파라미터 조합과 검증 MAPE를 확인합니다.
print('선택한 조합:', best_params)
print('검증 MAPE(%):', round(number=grid_table.loc[grid_table.index[0], '검증 MAPE(%)'], ndigits=3))

**해석**
- 하이퍼파라미터 조합에 따라 검증 MAPE가 조금씩 달라집니다.
  - `max_features`가 클수록 훈련 MAPE가 검증 MAPE보다 낮아 과적합 경향이 보이지만, 값을 줄이면 나무끼리 서로 달라져 평균의 효과가 커집니다.
  - `min_samples_leaf`를 키우면 리프 노드가 커져 예측이 매끄러워집니다.

In [ ]:
# 선택한 하이퍼파라미터로 랜덤 포레스트 모델을 생성합니다.
best_rf = RandomForestRegressor(n_estimators=300, random_state=SEED, **best_params)

# 훈련셋으로 랜덤 포레스트를 학습합니다.
best_rf.fit(X=X_train, y=y_train)

# 랜덤 포레스트를 models에 추가합니다.
models['랜덤 포레스트(그리드 서치)'] = best_rf

## 7. 최종 모델 선택

**질문**: 다섯 모델의 검증 성능을 비교하면 어느 모델이 가장 좋은가?

In [ ]:
# 모델별 검증 성능을 계산하고 valid_table에 할당합니다.
valid_table = compare(models, X_valid, y_valid)

# valid_table을 소수점 3자리까지 반올림해서 확인합니다.
valid_table.round(decimals=3)

In [ ]:
# 검증 MAPE가 가장 낮은 모델의 이름을 best_name에 할당합니다.
best_name = valid_table['MAPE(%)'].idxmin()

# 최종 모델의 이름을 확인합니다.
print('최종 모델:', best_name)

In [ ]:
# 최종 모델을 best_model에 할당합니다.
best_model = models[best_name]

In [ ]:
# 최종 모델과 점포별 평균의 검증 성능을 소수점 3자리까지 반올림해서 비교합니다.
pd.DataFrame(data={'점포별 평균': score(y=y_valid, pred=valid_store_pred), 
                   best_name: evaluate(best_model, X_valid, y_valid)}) \
    .T.round(decimals=3)

**해석**
- 최종 모델은 **검증셋 성능** 기준으로 선택합니다.
  - 시험셋을 보면서 고르면 시험셋에 맞춘 선택이 되어 실제 성능보다 좋게 보고하게 됩니다.
- 그리드 서치한 랜덤 포레스트가 검증 MAPE뿐 아니라 RMSE와 MAE도 가장 낮습니다.
  - 랜덤 포레스트는 여러 나무의 예측을 평균하므로 나무 하나보다 매끄럽게 예측합니다.
  - 아울러 선형 회귀가 잡지 못하는 관계(예: 행사일수가 늘 때 예측 매출 증가 폭이 대형 점포에서 더 큼, 12절)도 반영합니다.
- 점포별 평균과 비교하면 랜덤 포레스트의 검증 MAPE는 약 0.19%p 낮지만, RMSE는 점포별 평균이 조금 더 작습니다.
  - 검증 기간에는 점포의 평소 수준을 넘어서는 개선 폭이 크지 않습니다.
  - 미리 정한 선택 기준(검증 MAPE)에 따라 랜덤 포레스트를 최종 모델로 두고, 시험셋에서 다시 비교합니다.
- 다만 선형 회귀도 로그 변환처럼 데이터 구조에 맞게 변수를 처리하면 이 차이가 크게 줄어듭니다.
  - 성능이 비슷하면 설명하기 쉬운 모델을 고르는 것도 좋은 선택입니다.

## 8. 데이터 누수 확인

**질문**: 2절에서 제외한 `transactions`(구매 건수)를 입력변수에 넣으면 검증 성능이 어떻게 되는가?

In [ ]:
# 훈련셋과 검증셋의 입력변수 행렬에 구매 건수를 추가하고 X_train_leak, X_valid_leak에 할당합니다.
X_train_leak = X_train.assign(transactions=train_df['transactions'])
X_valid_leak = X_valid.assign(transactions=valid_df['transactions'])

# 그리드 서치로 고른 하이퍼파라미터로 랜덤 포레스트 모델을 생성합니다.
rf_leak = RandomForestRegressor(n_estimators=300, random_state=SEED, **best_params)

# 구매 건수를 추가한 훈련셋으로 랜덤 포레스트를 학습합니다.
rf_leak.fit(X=X_train_leak, y=y_train)

In [ ]:
# 두 모형의 검증 성능을 소수점 3자리까지 반올림해서 비교합니다.
pd.DataFrame(data={'랜덤 포레스트(그리드 서치)': evaluate(best_rf, X_valid, y_valid), 
                   '랜덤 포레스트(데이터 누수)': evaluate(rf_leak, X_valid_leak, y_valid)}) \
    .T.round(decimals=3)

**해석**
- `transactions`를 넣으면 MAPE가 크게 낮아집니다.
  - 숫자만 보면 훨씬 좋은 모델입니다.
  - 다음 달 구매 건수는 예측 시점에 알 수 없으므로 이 모델은 **실제로 쓸 수 없습니다.**
- **모델의 성능이 갑자기 너무 좋아지면 누수부터 의심합니다.**

## 9. 시험셋 평가 (마지막에 한 번)

**질문**: 다섯 모델의 시험셋(2026년 7 ~ 10월) 성능은 검증 결과와 같은 순서인가?

In [ ]:
# 모델별 시험 성능을 계산하고 test_table에 할당합니다.
test_table = compare(models, X_test, y_test)

# test_table을 소수점 3자리까지 반올림해서 확인합니다.
test_table.round(decimals=3)

In [ ]:
# 최종 모델과 점포별 평균의 시험 성능을 소수점 3자리까지 반올림해서 비교합니다.
pd.DataFrame(data={'점포별 평균': score(y=y_test, pred=test_store_pred), 
                   best_name: evaluate(best_model, X_test, y_test)}) \
    .T.round(decimals=3)

**해석**
- 시험 성능이 검증 성능과 비슷하면, 검증으로 고른 모델이 새 기간에도 통한다는 뜻입니다.
  - 검증에서 고른 랜덤 포레스트가 시험에서도 다섯 지표 모두 가장 좋습니다.
- 가지치기한 회귀 나무는 랜덤 포레스트와 선형 회귀보다 오차가 큽니다.
  - 나무 하나보다 여러 나무의 평균이 안정적이라는 것을 다시 확인할 수 있습니다.
- 가지치기 전 나무는 시험셋에서도 기준 모델을 빼면 오차가 가장 큽니다.
  - 과적합된 모델은 새 기간에 약하다는 것을 다시 확인할 수 있습니다.
- 점포별 평균과 비교하면 시험셋에서는 랜덤 포레스트의 MAPE가 약 2.59%p 낮고(9.168% → 6.580%), RMSE와 MAE도 더 작습니다.
  - 점포별 평균은 달마다 달라지는 명절·영업일수·행사를 반영하지 못하는 반면, 모델은 이런 달력 정보를 함께 쓰기 때문으로 보입니다.

## 10. 예측 결과 시각화

**질문**: 시험셋에서 실제 매출과 예측 매출은 얼마나 가까운가?

In [ ]:
# 최종 모델로 시험셋의 매출을 예측하고 test_pred에 할당합니다.
test_pred = best_model.predict(X=X_test)

# 실제값과 예측값의 산점도에 실제 = 예측 기준선을 추가합니다.
plt.figure(figsize=(6, 6), dpi=100)
plt.scatter(x=y_test, y=test_pred, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], c='red', label='실제 = 예측')
plt.title(label=f'시험셋: 실제 vs 예측 ({best_name})')
plt.xlabel(xlabel='실제 월 매출 (백만 원)')
plt.ylabel(ylabel='예측 월 매출 (백만 원)')
plt.legend()
plt.show()

**해석**
- 점이 빨간 선에 가까울수록 정확한 예측입니다.
- 점포 규모가 커질수록 점이 선에서 더 멀리 퍼지는데, 매출이 큰 점포는 금액 기준 오차도 크기 때문입니다.

**질문**: 면적이 가장 큰 점포와 가장 작은 점포의 2026년 월별 실제·예측 매출을 비교합니다.

In [ ]:
# 면적이 가장 큰 점포의 ID를 big_store에 할당합니다.
big_store = train_df.loc[train_df['floor_area'].idxmax(), 'store_id']

# 면적이 가장 작은 점포의 ID를 small_store에 할당합니다.
small_store = train_df.loc[train_df['floor_area'].idxmin(), 'store_id']

# 2026년(검증셋과 시험셋) 데이터와 입력변수 행렬을 위아래로 붙여서 df_2026, X_2026에 할당합니다.
df_2026 = pd.concat(objs=[valid_df, test_df], ignore_index=True)
X_2026 = pd.concat(objs=[X_valid, X_test], ignore_index=True)

# 두 점포의 2026년 월별 실제 매출과 예측 매출을 선 그래프로 비교합니다.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4), dpi=100)
for ax, store_id in zip(axes, [big_store, small_store]):
    mask = df_2026['store_id'].eq(other=store_id)
    ax.plot(df_2026.loc[mask, 'ym'], df_2026.loc[mask, 'sales'], marker='o', label='실제')
    ax.plot(df_2026.loc[mask, 'ym'], best_model.predict(X=X_2026.loc[mask, :]), marker='s', ls='--', label='예측')
    ax.set(title=f'{store_id} 월 매출: 실제 vs 예측 (2026년)', xlabel='연월', ylabel='월 매출 (백만 원)')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
plt.tight_layout()
plt.show()

**해석**
- 예측선은 실제선보다 훨씬 완만하게 움직입니다.
  - 모델은 점포의 평소 매출 수준은 잘 잡지만, 달마다 생기는 큰 오르내림까지는 따라가지 못합니다.
  - 이 흔들림의 일부는 날씨나 우연에 의한 것이라 예측 시점의 정보로는 맞히기 어렵습니다.
- 예측값은 '평소 수준의 기준선'으로 쓰고, 실제와의 차이가 크면 원인을 확인하는 방식으로 활용합니다.

## 11. 특성 중요도: 불순도 감소와 순열 중요도

**질문**: 랜덤 포레스트가 나무를 만들 때 많이 사용한 변수는 무엇인가? (불순도 감소 기준)

In [ ]:
# 랜덤 포레스트의 입력변수별 불순도 감소 중요도를 막대 그래프로 시각화합니다.
plt.figure(figsize=(6, 6), dpi=100)
hds.plot.feature_importance(model=best_rf)
plt.show()

**해석**
- 면적·품목 수·주차면수·유형처럼 **점포 규모**를 나타내는 변수가 불순도 감소 중요도 대부분을 차지합니다.
  - 점포 간 매출 차이가 워낙 크기 때문입니다.
- 선형 회귀 실습에서 품목 수는 다중공선성 때문에 뺐습니다.
  - 서로 비슷한 정보를 가진 변수들은 나무가 분할할 때 번갈아 선택되면서 **중요도를 나눠 가집니다.**
- **불순도 감소 중요도는 '예측에 많이 쓰였다'는 뜻이지 '원인이다'라는 뜻이 아닙니다.**
  - 주차장을 늘리면 매출이 오른다는 근거가 아닙니다.
- 효과의 크기와 방향(행사일 1일당 몇 % 등)이 궁금하면 선형 회귀의 계수를 봐야 합니다.

**질문**: 한 변수의 값을 무작위로 섞으면 검증 MAPE가 얼마나 높아지는가? (순열 중요도)

In [ ]:
# 검증셋에서 변수마다 값을 10번씩 섞어 MAPE 증가량을 계산합니다.
# [참고] permutation_importance()는 sklearn.inspection 모듈에 포함되어 있습니다.
# [참고] scoring='neg_mean_absolute_percentage_error'는 MAPE에 음수를 붙인 점수라서, 점수 감소량이 곧 MAPE 증가량입니다.
perm = permutation_importance(estimator=best_rf, X=X_valid, y=y_valid, scoring='neg_mean_absolute_percentage_error', 
                              n_repeats=10, random_state=SEED)

# 변수별 MAPE 증가량(%p)의 평균과 표준편차를 perm_table에 할당하고 평균이 작은 순서로 정렬합니다.
# [참고] barh()는 아래에서부터 막대를 그리므로 평균이 큰 변수가 위에 표시됩니다.
# [참고] MAPE를 %로 나타내기 위해 100을 곱합니다.
perm_table = pd.DataFrame(data={'평균': perm.importances_mean * 100, '표준편차': perm.importances_std * 100}, 
                          index=X_valid.columns) \
    .sort_values(by='평균')

# 순열 중요도를 가로 막대 그래프로 시각화하고, 막대 오른쪽에 값을 표시합니다.
# [참고] 0보다 조금 작은 값도 라벨이 변수 이름과 겹치지 않도록 x 좌표를 0 이상으로 지정합니다.
plt.figure(figsize=(6, 6), dpi=100)
plt.barh(y=perm_table.index, width=perm_table['평균'], color='silver')
for name, value in perm_table['평균'].items():
    plt.text(x=max(value, 0), y=name, s=f' {value:.3f}', va='center', fontsize=8, fontweight='bold')
plt.xlim(right=perm_table['평균'].max() * 1.2)
plt.title(label='순열 중요도 (검증셋)')
plt.xlabel(xlabel='값을 섞었을 때 MAPE 증가량(%p)')
plt.show()

**해석**
- 순열 중요도는 '이 변수의 순서를 섞으면(값이 엉망이 되면) 예측이 얼마나 나빠지는가'를 **검증셋**에서 직접 잽니다.
  - 순열 중요도에서 `floor_area`(면적)이 가장 큽니다.
  - 면적 값을 섞으면 검증 MAPE가 약 12.95%p 높아진다는 의미입니다.
- `region_수도권`(수도권 여부)는 불순도 감소 중요도보다 순열 중요도에서 순위가 높습니다.
  - `store_type_소형`·`store_type_중형`과 순서를 맞바꿨습니다.
- [참고] 점포 유형 더미 한 열만 섞어도 '어느 유형에도 속하지 않는 점포'처럼 실제로 있을 수 없는 조합이 생기므로, 더미 변수의 순열 중요도는 대략적인 참고로만 봅니다.
- `holiday_season`(명절 시즌)과 `holiday_days`(평일 공휴일 수)는 불순도 감소 중요도에서는 중간 정도였지만, 순열 중요도는 0 근처입니다.
  - 훈련셋에서 분할에 쓰였더라도, 값을 섞었을 때 검증 MAPE는 거의 변하지 않았다는 뜻입니다.
- 순열 중요도에서 높은 순위를 보이는 변수들이 예측 매출을 **어떻게** 바꾸는지는 다음 절의 부분 의존도로 확인합니다.

## 12. 부분 의존도(PDP)로 예측 방식 이해하기

**질문**: 순열 중요도 상위 4개 수치형 변수가 바뀔 때 평균 예측 매출은 어떻게 변하는가?

In [ ]:
# 더미 변수를 제외하고 순열 중요도 상위 4개 수치형 변수명을 리스트로 생성합니다.
# [참고] 더미 변수 한 열만 0 또는 1로 바꾸면 '소형이면서 대형인 점포'처럼 실제로 있을 수 없는 조합이 생기므로 부분 의존도에서 제외합니다.
perm_order = perm_table.sort_values(by='평균', ascending=False).index
top4_features = [col for col in perm_order if col in num_features][:4]

# top4_features를 출력합니다.
print('순열 중요도 상위 4개 수치형 변수:', top4_features)

In [ ]:
# 2행 2열의 그래프 영역을 생성합니다.
fig, ax = plt.subplots(figsize=(12, 8), dpi=100, nrows=2, ncols=2)

# 순열 중요도 상위 4개 변수의 부분 의존도를 그립니다.
# [참고] PartialDependenceDisplay 클래스는 sklearn.inspection 모듈에 포함되어 있습니다.
# [참고] kind='average'는 평균 부분 의존도를, kind='individual'은 개별 부분 의존도를 의미합니다.
# [참고] grid_resolution은 x축 눈금 개수, n_cols는 열 개수, ax는 그래프 영역을 지정합니다.
PartialDependenceDisplay.from_estimator(estimator=best_rf, X=X_valid, features=top4_features, kind='average', 
                                        grid_resolution=30, n_cols=2, ax=ax)
plt.tight_layout()
plt.show()

**해석**
- 부분 의존도 플롯의 세로축은 평균 예측 매출(백만 원)입니다.
- 순열 중요도 상위 4개 수치형 변수별 부분 의존도 플롯 결과는 다음과 같습니다.

  | 변수 | 부분 의존도 모양 | 해석 |
  |---|---|---|
  | `floor_area`<br>(면적) | 계단 모양으로 꾸준히 올라감 | 면적이 넓을수록 예측 매출이 높음 |
  | `sku_count`<br>(품목 수) | 계단 모양으로 올라감 | 품목 수가 많을수록 예측 매출이 높음 |
  | `parking`<br>(주차면수) | 계단 모양으로 올라감 | 주차면수가 많을수록 예측 매출이 높음 |
  | `event_days`<br>(행사일수) | 완만하게 올라가다 10일 이후 거의 평평함 | 행사일수가 많은 달일수록 예측 매출이 조금 높음 |

  [참고] 면적·품목 수·주차면수 곡선이 계단 모양인 것은 나무가 변수의 구간을 나눠 같은 값을 예측하기 때문입니다.
- 면적·품목 수·주차면수는 서로 상관이 높습니다.
  - 품목 수의 부분 의존도는 면적을 그대로 둔 채 품목 수만 바꾸므로, '면적은 작은데 품목 수는 아주 많은 점포'처럼 실제로 드문 조합의 예측도 평균에 들어갑니다.
  - 면적의 부분 의존도도 점포 유형은 그대로 둔 채 면적만 바꾸므로, '면적은 아주 넓은데 소형인 점포'처럼 자료에 없는 조합이 평균에 들어갑니다.
  - 네 곡선 모두 '모델이 이 변수를 어떻게 쓰는가'로만 읽고, 실제 점포끼리의 매출 차이로 읽지 않습니다.

**질문**: 행사일수가 늘 때 예측 매출의 변화가 점포 규모에 따라 다른가? (2변수 부분 의존도)

In [ ]:
# 1행 1열의 그래프 영역을 생성합니다.
fig, ax = plt.subplots(figsize=(6, 6), dpi=100)

# 면적과 행사일수의 2변수 부분 의존도를 등고선으로 그립니다.
# [참고] 2변수는 계산량이 많아 grid_resolution을 작게 지정합니다.
PartialDependenceDisplay.from_estimator(estimator=best_rf, X=X_valid, features=[('floor_area', 'event_days')],
                                        contour_kw={'alpha': 0.5}, grid_resolution=15, ax=ax)
plt.show()

In [ ]:
# 점포 유형별 검증셋 행사일수의 최솟값과 최댓값을 확인합니다.
# [참고] 다음 셀에서 비교할 행사일수(3일, 12일)가 소형 점포와 대형 점포 모두에서 실제로 관측된 값인지 확인합니다.
valid_df.groupby(by='store_type')['event_days'].agg(func=['min', 'max'])

In [ ]:
# 실제 소형 점포와 대형 점포의 검증셋 행에서 행사일수만 3일과 12일로 바꾼 평균 예측 매출을 계산합니다.
# [참고] 면적, 점포 유형, 품목 수, 주차면수 등 나머지 변수는 각 점포의 실제 값을 그대로 둡니다.
did_rows = []
for store_type in ['소형', '대형']:
    X_group = X_valid.loc[valid_df['store_type'] == store_type, :]
    for days in [3, 12]:
        pred = best_rf.predict(X=X_group.assign(event_days=days))
        did_rows.append({'점포 유형': store_type, '행사일수': days, '평균 예측 매출': pred.mean()})

# 점포 유형을 행, 행사일수를 열로 바꾸고 did_table에 할당합니다.
did_table = pd.DataFrame(data=did_rows).pivot(index='점포 유형', columns='행사일수', values='평균 예측 매출') \
    .reindex(index=['소형', '대형'])

In [ ]:
# 점포 유형별로 행사일수가 3일에서 12일로 늘 때의 평균 예측 매출 변화와 변화율(%)을 추가합니다.
did_table['변화(12일 − 3일)'] = did_table[12] - did_table[3]
did_table['변화율(%)'] = did_table['변화(12일 − 3일)'] / did_table[3] * 100

# 대형 점포의 변화에서 소형 점포의 변화를 뺀 차이의 차이를 추가합니다.
did_table['차이의 차이'] = did_table.loc['대형', '변화(12일 − 3일)'] - did_table.loc['소형', '변화(12일 − 3일)']

# did_table을 소수점 첫째 자리까지 반올림해서 확인합니다.
did_table.round(decimals=1)

**해석**
- 2변수 부분 의존도 플롯을 읽는 방법입니다.
  - 색과 등고선은 면적과 행사일수가 각각 그 값일 때의 평균 예측 매출(백만 원)입니다.
  - 색이 밝을수록(등고선 숫자가 클수록) 평균 예측 매출이 높습니다.
- 등고선이 거의 세로로 서 있습니다.
  - 평균 예측 매출은 주로 면적에 따라 달라지고, 행사일수에 따라서는 크게 달라지지 않습니다.
- [참고] 2변수 부분 의존도는 모든 행의 면적을 같은 값으로 바꾸고 점포 유형·품목 수·주차면수는 그대로 둡니다.
  - 그 결과 '면적은 아주 넓은데 소형인 점포'처럼 자료에 없는 조합이 생깁니다.
  - 그래프는 전체 모양만 보고, 면적별 효과의 크기는 실제 점포 안에서 따로 비교합니다.
- 실제 소형 점포와 대형 점포에서 행사일수만 3일에서 12일로 바꿨을 때 평균 예측 매출의 변화는 다음과 같습니다.
  - 두 행사일수 모두 소형 점포와 대형 점포에서 실제로 관측된 범위 안의 값입니다.
  - 소형 점포는 약 73백만 원(약 3.2%), 대형 점포는 약 248백만 원(약 4.7%) 늘어납니다.
  - 두 변화의 차이(차이의 차이)는 약 176백만 원입니다.
  - 매출 금액으로도 비율로도 대형 점포에서 더 크게 늘어나는 것으로 예측합니다.
- 이 차이는 '모델이 소형 점포와 대형 점포의 행사일수에 다르게 반응한다'는 뜻입니다.
  - 두 집단은 면적뿐 아니라 점포 유형·품목 수·주차면수가 모두 다르므로, 면적만의 상호작용이라고 할 수 없습니다.
  - 행사일을 늘렸을 때 실제로 매출이 이만큼 늘어난다는 인과 효과도 아닙니다.
- [참고] 상호작용이 있는지는 어느 척도(매출 금액, 로그 매출 등)로 보느냐에 따라 달라지므로, 척도를 함께 밝힙니다.

## 정리

- 예측 시점에 알 수 없는 변수(구매 건수, 객단가, 실제 날씨)를 입력변수에서 제외했습니다.
  - 최종 모형에 구매 건수를 넣으면 성능이 비정상적으로 좋아지는 누수를 확인했습니다.
- 데이터를 시간 순서로 훈련·검증·시험으로 나누고, 하이퍼파라미터와 최종 모델은 검증셋으로 선택했습니다.
- 모든 매출을 훈련셋 평균으로 예측하는 기준 모델은 MAPE가 약 35%였고, 선형 회귀와 머신러닝 모델은 모두 이보다 크게 좋았습니다.
- 점포마다 2025년 월평균으로 예측하는 점포별 평균도 검증 MAPE가 약 6.2%로 강한 기준선이었습니다.
  - 랜덤 포레스트는 점포별 평균보다 검증 MAPE와 MAE가 조금 낮았지만, 검증 RMSE와 MSE는 조금 높았습니다.
  - 시험셋에서는 다섯 지표 모두 점포별 평균보다 좋았습니다.
- 끝까지 자란 회귀 나무는 훈련셋을 외워 과적합되었고, 사후 가지치기로 검증 성능을 높였습니다.
- 그리드 서치한 랜덤 포레스트는 기준 모델, 선형 회귀, 회귀 나무보다 검증과 시험 모두 모든 지표에서 좋았습니다.
  - 선형 회귀도 로그 변환처럼 데이터 구조에 맞게 변수를 처리하면 이 차이를 크게 줄일 수 있습니다.
- 순열 중요도로 검증셋에서도 점포 규모 변수(면적, 품목 수, 점포 유형, 주차면수)가 예측에 가장 중요하다는 것을 확인했습니다.
- 부분 의존도로 예측 매출은 주로 면적에 따라 달라진다는 것을 확인했습니다.
  - 실제 점포 안에서 행사일수만 바꾸면, 모델은 대형 점포의 예측 매출을 소형 점포보다 금액과 비율 모두 더 크게 올립니다.
- 특성 중요도와 부분 의존도는 예측 기여도일 뿐 인과 효과가 아닙니다.

## 현업 전달 포인트

통계 용어 없이 설명하면 다음과 같습니다.

- 다음 달 점포 매출을 실제 매출 대비 평균 약 6.6%의 오차로 예측할 수 있습니다.
  - 발주·인력 계획의 기준값으로 쓸 수 있습니다.
- 점포마다 작년 월평균만 써도 꽤 정확하게 맞힐 수 있으므로, 모델은 이 단순한 방법보다 나은지 확인하고 씁니다.
  - 최근 4개월에는 모델의 오차가 약 6.6%로 작년 월평균(약 9.2%)보다 작았습니다.
  - 다만 그 전 6개월에는 두 방법의 정확도가 비슷했으므로, 계속 비교하며 씁니다.
- 모델 성능은 학습과 설정 선택에 쓰지 않은 최근 4개월 데이터로 평가한 값입니다.
  - 실제로 사용할 때의 정확도에 가깝습니다.
- 이 모델은 '얼마나 팔릴지'를 맞히는 용도입니다.
  - '무엇을 바꾸면 매출이 오를지'는 별도 분석(회귀분석, 실험)이 필요합니다.

## End of Document